# 183 — Attention Head Clustering

Cluster heads by functional behavior: output direction clustering,
logit profile clustering, cooperation groups, specialization spectrum,
and redundancy analysis.

## Why This Matters

Attention head clustering reveals how heads route information between positions. Understanding attention mechanics is central to reverse-engineering transformer circuits, as attention patterns determine what information each component can access and transform.

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis
- [Olsson et al. (2022) "In-context Learning and Induction Heads"](https://arxiv.org/abs/2209.11895) — Discovers induction heads and training phase transitions

In [ ]:
import jax
import jax.numpy as jnp
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.attention_head_clustering import (
    head_output_clustering,
    head_logit_profile_clustering,
    head_cooperation_groups,
    head_specialization_spectrum,
    head_redundancy_analysis,
)

In [ ]:
cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.arange(15)

In [ ]:
result = head_output_clustering(model, tokens, n_clusters=4)
print(f"Clustered {result['n_total']} heads into {result['n_clusters']} groups:\n")
for c in result['clusters']:
    members = ', '.join(f'L{m["layer"]}H{m["head"]}' for m in c['members'])
    print(f"  Cluster {c['cluster']} ({c['n_members']}): {members}")

In [ ]:
result = head_logit_profile_clustering(model, tokens, n_clusters=3)
for c in result['clusters']:
    members = ', '.join(f'L{m["layer"]}H{m["head"]}' for m in c['members'])
    toks = ', '.join(str(t) for t in c['top_promoted_tokens'][:5])
    print(f"Cluster {c['cluster']} ({c['n_members']}): promotes [{toks}...]")
    print(f"  Members: {members}\n")

In [ ]:
result = head_cooperation_groups(model, tokens, threshold=0.3)
print(f"Cooperating: {result['n_cooperating']}  Opposing: {result['n_opposing']}\n")
print("Top cooperating pairs:")
for p in result['cooperating_pairs'][:5]:
    a = f"L{p['head_a']['layer']}H{p['head_a']['head']}"
    b = f"L{p['head_b']['layer']}H{p['head_b']['head']}"
    print(f"  {a} <-> {b}: cos={p['cosine']:.3f}")
print("\nTop opposing pairs:")
for p in result['opposing_pairs'][:5]:
    a = f"L{p['head_a']['layer']}H{p['head_a']['head']}"
    b = f"L{p['head_b']['layer']}H{p['head_b']['head']}"
    print(f"  {a} <-> {b}: cos={p['cosine']:.3f}")

In [ ]:
result = head_specialization_spectrum(model, tokens)
for h in sorted(result['per_head'], key=lambda x: -x['logit_concentration']):
    print(f"L{h['layer']}H{h['head']}: selectivity={h['attention_selectivity']:.3f}  "
          f"norm={h['output_norm']:.4f}  logit_conc={h['logit_concentration']:.4f}")

In [ ]:
result = head_redundancy_analysis(model, tokens, threshold=0.5)
print(f"Redundant pairs (>{0.5} cosine): {result['n_redundant']}/{result['n_total_pairs']}  "
      f"({result['redundancy_fraction']:.1%})\n")
for p in result['redundant_pairs'][:10]:
    a = f"L{p['head_a']['layer']}H{p['head_a']['head']}"
    b = f"L{p['head_b']['layer']}H{p['head_b']['head']}"
    print(f"  {a} ~ {b}: cos={p['cosine']:.3f}")